# YouTube Trending Videos EDA

Notebook-based cleaning and inspection for the YouTube trending dataset.

## Introduction

This notebook loads the raw trending-video dataset, inspects the source schema, and cleans the data in memory so the analysis can be reproduced step by step. The cleaning work is kept visible in separate cells instead of being hidden inside one large block.

## Problem Statement

Content creators and marketing teams need to understand which YouTube video characteristics are associated with trending performance. This analysis cleans the raw dataset and prepares it for studying trending frequency, engagement, timing, categories, and tags.

## Objectives

- Inspect the raw data structure and quality.
- Clean data types, text fields, and placeholder values.
- Preserve row-level trending data and create a per-video view.
- Engineer the requested features for downstream analysis.
- Document the cleaning decisions inline in markdown.

## Exploratory Data Analysis

### Data Info

The first cells load the raw CSV and inspect the schema, datatypes, descriptive statistics, missing values, and sample rows.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_CANDIDATES = [
    Path("data/raw/youtube_trending_raw.csv"),
    Path("../data/raw/youtube_trending_raw.csv"),
]
for candidate in RAW_CANDIDATES:
    if candidate.exists():
        RAW_PATH = candidate
        break
else:
    raise FileNotFoundError("Could not find data/raw/youtube_trending_raw.csv")

raw_df = pd.read_csv(RAW_PATH)
raw_df.head()

,index,video_id,trending_date,title,channel_title,category_id,publish_date,time_frame,published_day_of_week,publish_country,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed
0,0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,13/11/2017,17:00 to 17:59,Monday,US,SHANtell martin,748374,57527,2966,15954,False,False,False
1,1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,13/11/2017,7:00 to 7:59,Monday,US,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,False,False,False
2,2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,12/11/2017,19:00 to 19:59,Sunday,US,"racist superman|""rudy""""|""""mancuso""""|""""king""""|""...",3191434,146033,5339,8181,False,False,False
3,3,puqaWrEC7tY,17.14.11,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,13/11/2017,11:00 to 11:59,Monday,US,"rhett and link|""gmm""""|""""good mythical morning""...",343168,10172,666,2146,False,False,False
4,4,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,12/11/2017,18:00 to 18:59,Sunday,US,"ryan|""higa""""|""""higatv""""|""""nigahiga""""|""""i dare ...",2095731,132235,1989,17518,False,False,False


In [2]:
print('shape:', raw_df.shape)
print('\ncolumns:')
print(raw_df.columns.tolist())
print('\ndtypes:')
print(raw_df.dtypes)

print('\ninfo:')
raw_df.info()

print('\ndescribe:')
display(raw_df.describe(include='all').T)

shape: (161470, 18)

columns:
['index', 'video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_date', 'time_frame', 'published_day_of_week', 'publish_country', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed']

dtypes:
index                     int64
video_id                    str
trending_date               str
title                       str
channel_title               str
category_id               int64
publish_date                str
time_frame                  str
published_day_of_week       str
publish_country             str
tags                        str
views                     int64
likes                     int64
dislikes                  int64
comment_count             int64
comments_disabled          bool
ratings_disabled           bool
video_error_or_removed     bool
dtype: object

info:
<class 'pandas.DataFrame'>
RangeIndex: 161470 entries, 0 to 161469
Data columns (total 1

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
index,161470.0,NaN,NaN,NaN,80734.5,46612.51832,0.0,40367.25,80734.5,121101.75,161469.0
video_id,161470,55886,#NAME?,1799,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,161470,205,17.14.11,800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,161470,56905,Childish Gambino - This Is America (Official V...,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,161470,12361,The Late Show with Stephen Colbert,653,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,161470.0,NaN,NaN,NaN,19.461151,7.432001,1.0,15.0,23.0,24.0,44.0
publish_date,161470,471,20/12/2017,1307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time_frame,161470,24,16:00 to 16:59,16477,NaN,NaN,NaN,NaN,NaN,NaN,NaN
published_day_of_week,161470,7,Friday,28622,NaN,NaN,NaN,NaN,NaN,NaN,NaN
publish_country,161470,4,US,40949,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
missing_counts = raw_df.isna().sum().sort_values(ascending=False)
display(missing_counts[missing_counts > 0].to_frame('missing_count'))

print('sample rows:')
display(raw_df.sample(5, random_state=42))

,missing_count


sample rows:


,index,video_id,trending_date,title,channel_title,category_id,publish_date,time_frame,published_day_of_week,publish_country,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed
23971,23971,tEmodtG8NV0,18.16.03,I TOOK A COMPATIBILITY TEST w/ MY SOUL MATE!,Gabbie Hanna,26,14/03/2018,23:00 to 23:59,Wednesday,US,"the gabbie show|""thegabbieshow""""|""""soul-mate""""...",1267850,72128,2858,8151,False,False,False
102376,102376,rD7dFsO_XwI,18.08.03,TUTO DE MES STYLES PREFÃ‰RÃ‰S DE TURBAN ( FACI...,Honey Shay,24,07/03/2018,18:00 to 18:59,Wednesday,FRANCE,"honeyshay""|""tuto""|""turban""|""tuto foulard""|""tut...",18526,2651,37,293,False,False,False
47446,47446,ip1FhJhM7vg,17.16.12,Bryson Tiller on being in a dark place after T...,TimWestwoodTV,10,12/12/2017,16:00 to 16:59,Tuesday,GB,"tim|""westwood""|""tv""|""freestyle""|""crib""|""sessio...",12316,292,5,43,False,False,False
159272,159272,nfuXbnmdpRE,18.04.06,"$40 Gym Vs. $10,000 Gym",BuzzFeedBlue,22,03/06/2018,15:00 to 15:59,Sunday,CANADA,"BuzzFeed|""BuzzFeed Worth It""|""$$""|""Worth It Li...",1533833,31892,2062,2930,False,False,False
54821,54821,ic1kMoM_kPw,18.24.01,Dylan Farrow details her sexual assault allega...,CBS This Morning,25,18/01/2018,13:00 to 13:59,Thursday,GB,"video|""cbs""|""news""|""Dylan Farrow""|""sexual assa...",247334,1121,887,1144,False,False,False


### Data Handling

Cleaning decisions used in this notebook:

- String placeholders such as `#NAME?`, `[none]`, `none`, `null`, and `nan` are normalized instead of being treated as real values.
- Text columns are stripped of surrounding whitespace.
- `tags` is normalized into a lowercase, deduplicated pipe-delimited string for frequency analysis.
- Date and numeric fields are parsed into proper types.
- Invalid numeric values are coerced to missing rather than silently clipped.
- The row-level dataset is preserved for time-based analysis, and a separate per-video view is created from the latest trending snapshot.
- Missing or malformed text values are labeled as `unknown` where that is more useful than dropping rows.
- The notebook keeps all cleaned data in memory only; it does not write a cleaned export file.

In [6]:
TEXT_PLACEHOLDERS = {'#name?', '[none]', 'none', 'null', 'nan', ''}
BOOLEAN_COLUMNS = ['comments_disabled', 'ratings_disabled', 'video_error_or_removed']
NUMERIC_COLUMNS = ['category_id', 'views', 'likes', 'dislikes', 'comment_count']
CATEGORY_LABELS = {
    1: 'Film & Animation',
    2: 'Autos & Vehicles',
    10: 'Music',
    15: 'Pets & Animals',
    17: 'Sports',
    18: 'Short Movies',
    19: 'Travel & Events',
    20: 'Gaming',
    22: 'People & Blogs',
    23: 'Comedy',
    24: 'Entertainment',
    25: 'News & Politics',
    26: 'Howto & Style',
    27: 'Education',
    28: 'Science & Technology',
    29: 'Nonprofits & Activism',
    30: 'Movies',
    43: 'Shows',
}

def normalize_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in ['video_id', 'title', 'channel_title', 'time_frame', 'tags']:
        if column in df.columns:
            series = df[column].astype('string').str.strip()
            if column == 'tags':
                df[column] = series.fillna('')
            else:
                df[column] = series.mask(series.str.lower().isin(TEXT_PLACEHOLDERS), 'unknown')
    if 'publish_country' in df.columns:
        series = df['publish_country'].astype('string').str.strip()
        df['publish_country'] = series.mask(series.str.lower().isin(TEXT_PLACEHOLDERS), 'unknown').str.upper()
    if 'published_day_of_week' in df.columns:
        series = df['published_day_of_week'].astype('string').str.strip()
        df['published_day_of_week'] = series.mask(series.str.lower().isin(TEXT_PLACEHOLDERS), 'unknown').str.title()
    return df

def coerce_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in NUMERIC_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce')
    if 'category_id' in df.columns:
        df['category_id'] = df['category_id'].astype('Int64')
    for column in ['views', 'likes', 'dislikes', 'comment_count']:
        if column in df.columns:
            df[column] = df[column].astype('Int64')
            df.loc[df[column] < 0, column] = pd.NA
    return df

def coerce_boolean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    truthy = {'true', '1', 'yes', 'y', 't'}
    falsy = {'false', '0', 'no', 'n', 'f'}
    for column in BOOLEAN_COLUMNS:
        if column not in df.columns:
            continue
        series = df[column].astype('string').str.strip().str.lower()
        mapped = series.map(lambda value: True if value in truthy else False if value in falsy else pd.NA)
        df[column] = mapped.astype('boolean')
    return df

def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'trending_date' in df.columns:
        df['trending_date'] = pd.to_datetime(df['trending_date'].astype('string'), format='%y.%d.%m', errors='coerce')
    if 'publish_date' in df.columns:
        df['publish_date'] = pd.to_datetime(df['publish_date'].astype('string'), format='%d/%m/%Y', errors='coerce')
    return df

def parse_time_frame_start_hour(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r'(\d{1,2}):\d{2}\s*to\s*(\d{1,2}):\d{2}', str(value))
    if not match:
        return pd.NA
    return int(match.group(1))

def clean_tags(value):
    if pd.isna(value):
        return ''
    cleaned = []
    seen = set()
    for raw_tag in str(value).split('|'):
        tag = raw_tag.strip().strip('"').strip("'").strip().lower()
        tag = re.sub(r'\s+', ' ', tag)
        if not tag or tag in {'[none]', 'none', 'nan', 'null'}:
            continue
        if tag not in seen:
            seen.add(tag)
            cleaned.append(tag)
    return '|'.join(cleaned)

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'time_frame' in df.columns:
        df['published_hour'] = df['time_frame'].map(parse_time_frame_start_hour).astype('Int64')
    else:
        df['published_hour'] = pd.Series([pd.NA] * len(df), dtype='Int64')
    if {'trending_date', 'publish_date'}.issubset(df.columns):
        df['days_to_trend'] = (df['trending_date'] - df['publish_date']).dt.days.astype('Int64')
    else:
        df['days_to_trend'] = pd.Series([pd.NA] * len(df), dtype='Int64')
    views = df['views'].astype('float64') if 'views' in df.columns else pd.Series(dtype='float64')
    likes = df['likes'].astype('float64') if 'likes' in df.columns else pd.Series(dtype='float64')
    dislikes = df['dislikes'].astype('float64') if 'dislikes' in df.columns else pd.Series(dtype='float64')
    comments = df['comment_count'].astype('float64') if 'comment_count' in df.columns else pd.Series(dtype='float64')
    views_safe = views.where(views > 0)
    like_total = likes + dislikes
    df['engagement_rate'] = (likes + comments) / views_safe
    df['like_ratio'] = likes / like_total.where(like_total > 0)
    df['engagement_rate'] = df['engagement_rate'].replace([np.inf, -np.inf], np.nan)
    df['like_ratio'] = df['like_ratio'].replace([np.inf, -np.inf], np.nan)
    if {'publish_country', 'video_id'}.issubset(df.columns):
        df['unique_video_key'] = df['publish_country'].fillna('unknown').astype('string') + '::' + df['video_id'].fillna('unknown').astype('string')
    else:
        df['unique_video_key'] = 'unknown'
    if 'tags' in df.columns:
        df['tags_clean'] = df['tags'].map(clean_tags)
        df['tag_count'] = df['tags_clean'].map(lambda value: 0 if not value else len(value.split('|'))).astype('Int64')
    else:
        df['tags_clean'] = ''
        df['tag_count'] = pd.Series([0] * len(df), dtype='Int64')
    if 'category_id' in df.columns:
        df['category_label'] = df['category_id'].map(CATEGORY_LABELS).fillna('Unknown')
    else:
        df['category_label'] = 'Unknown'
    metric_issue_flag = pd.Series(False, index=df.index)
    for column in ['views', 'likes', 'dislikes', 'comment_count']:
        if column in df.columns:
            metric_issue_flag = metric_issue_flag | df[column].isna()
    if {'views', 'likes'}.issubset(df.columns):
        metric_issue_flag = metric_issue_flag | (df['likes'] > df['views'])
    if {'views', 'comment_count'}.issubset(df.columns):
        metric_issue_flag = metric_issue_flag | (df['comment_count'] > df['views'])
    df['metric_issue_flag'] = metric_issue_flag.astype('boolean')
    if {'unique_video_key', 'trending_date'}.issubset(df.columns):
        df['days_trending'] = df.groupby('unique_video_key')['trending_date'].transform('count').astype('Int64')
    else:
        df['days_trending'] = pd.Series([pd.NA] * len(df), dtype='Int64')
    return df

clean_df = raw_df.copy()

In [7]:
clean_df = normalize_text_columns(clean_df)
clean_df = coerce_numeric_columns(clean_df)
clean_df = coerce_boolean_columns(clean_df)
clean_df = parse_dates(clean_df)

display(clean_df.head())
display(clean_df.dtypes.to_frame('dtype'))

,index,video_id,trending_date,title,channel_title,category_id,publish_date,time_frame,published_day_of_week,publish_country,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed
0,0,2kyS6SvSYSE,2017-11-14,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13,17:00 to 17:59,Monday,US,SHANtell martin,748374,57527,2966,15954,False,False,False
1,1,1ZAPwfrtAFY,2017-11-14,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13,7:00 to 7:59,Monday,US,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,False,False,False
2,2,5qpjK5DgCt4,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12,19:00 to 19:59,Sunday,US,"racist superman|""rudy""""|""""mancuso""""|""""king""""|""...",3191434,146033,5339,8181,False,False,False
3,3,puqaWrEC7tY,2017-11-14,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13,11:00 to 11:59,Monday,US,"rhett and link|""gmm""""|""""good mythical morning""...",343168,10172,666,2146,False,False,False
4,4,d380meD0W0M,2017-11-14,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12,18:00 to 18:59,Sunday,US,"ryan|""higa""""|""""higatv""""|""""nigahiga""""|""""i dare ...",2095731,132235,1989,17518,False,False,False


,dtype
index,int64
video_id,string
trending_date,datetime64[us]
title,string
channel_title,string
category_id,Int64
publish_date,datetime64[us]
time_frame,string
published_day_of_week,string
publish_country,string


In [8]:
print('missing values after type coercion:')
display(clean_df.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

print('duplicate full rows:', int(clean_df.duplicated().sum()))
print('duplicate video/date rows:', int(clean_df.duplicated(subset=['video_id', 'trending_date']).sum()))

print('decision notes:')
print('- No rows are dropped at this stage because the source data has no true missing values in the core fields.')
print('- Placeholder text values were normalized to unknown or empty tags.')
print('- Invalid numeric values would be coerced to NA instead of being clipped.')

missing values after type coercion:


,missing_count
index,0
video_id,0
trending_date,0
title,0
channel_title,0
category_id,0
publish_date,0
time_frame,0
published_day_of_week,0
publish_country,0


duplicate full rows: 0
duplicate video/date rows: 19341
decision notes:
- No rows are dropped at this stage because the source data has no true missing values in the core fields.
- Placeholder text values were normalized to unknown or empty tags.
- Invalid numeric values would be coerced to NA instead of being clipped.


In [9]:
clean_df = add_engineered_features(clean_df)
display(clean_df[['video_id', 'trending_date', 'publish_date', 'published_hour', 'days_to_trend', 'engagement_rate', 'like_ratio', 'tag_count', 'days_trending', 'unique_video_key']].head())
display(clean_df[['tags', 'tags_clean']].head(5))

,video_id,trending_date,publish_date,published_hour,days_to_trend,engagement_rate,like_ratio,tag_count,days_trending,unique_video_key
0,2kyS6SvSYSE,2017-11-14,2017-11-13,17,1,0.098188,0.950970,1,7,US::2kyS6SvSYSE
1,1ZAPwfrtAFY,2017-11-14,2017-11-13,7,1,0.045431,0.940521,4,7,US::1ZAPwfrtAFY
2,5qpjK5DgCt4,2017-11-14,2017-11-12,19,2,0.048321,0.964729,23,7,US::5qpjK5DgCt4
3,puqaWrEC7tY,2017-11-14,2017-11-13,11,1,0.035895,0.938550,27,7,US::puqaWrEC7tY
4,d380meD0W0M,2017-11-14,2017-11-12,18,2,0.071456,0.985181,14,6,US::d380meD0W0M


,tags,tags_clean
0,SHANtell martin,shantell martin
1,"last week tonight trump presidency|""last week ...",last week tonight trump presidency|last week t...
2,"racist superman|""rudy""""|""""mancuso""""|""""king""""|""...",racist superman|rudy|mancuso|king|bach|racist|...
3,"rhett and link|""gmm""""|""""good mythical morning""...",rhett and link|gmm|good mythical morning|rhett...
4,"ryan|""higa""""|""""higatv""""|""""nigahiga""""|""""i dare ...",ryan|higa|higatv|nigahiga|i dare you|idy|rhpc|...


In [10]:
negative_counts = {column: int((clean_df[column] < 0).sum()) for column in ['views', 'likes', 'dislikes', 'comment_count']}
print('negative counts:', negative_counts)
print('likes greater than views:', int((clean_df['likes'] > clean_df['views']).sum()))
print('comments greater than views:', int((clean_df['comment_count'] > clean_df['views']).sum()))
print('metric issues flagged:', int(clean_df['metric_issue_flag'].sum()))
display(clean_df.loc[clean_df['metric_issue_flag']].head())

negative counts: {'views': 0, 'likes': 0, 'dislikes': 0, 'comment_count': 0}
likes greater than views: 0
comments greater than views: 0
metric issues flagged: 0


,index,video_id,trending_date,title,channel_title,category_id,publish_date,time_frame,published_day_of_week,publish_country,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed,published_hour,days_to_trend,engagement_rate,like_ratio,unique_video_key,tags_clean,tag_count,category_label,metric_issue_flag,days_trending


In [11]:
sorted_df = clean_df.sort_values(['unique_video_key', 'trending_date', 'publish_date'], ascending=[True, False, False], kind='mergesort')
video_level_df = sorted_df.drop_duplicates(subset=['unique_video_key'], keep='first').copy()
trend_bounds = clean_df.groupby('unique_video_key', as_index=False).agg(
    first_trending_date=('trending_date', 'min'),
    last_trending_date=('trending_date', 'max'),
)
video_level_df = video_level_df.merge(trend_bounds, on='unique_video_key', how='left')
display(video_level_df[['video_id', 'unique_video_key', 'trending_date', 'first_trending_date', 'last_trending_date', 'days_trending']].head())
print('row-level shape:', clean_df.shape)
print('video-level shape:', video_level_df.shape)
print('row-level trending date range:', clean_df['trending_date'].min(), 'to', clean_df['trending_date'].max())

,video_id,unique_video_key,trending_date,first_trending_date,last_trending_date,days_trending
0,--45ws7CEN0,CANADA::--45ws7CEN0,2018-06-12,2018-06-12,2018-06-12,1
1,--7vNbh4UNA,CANADA::--7vNbh4UNA,2018-04-16,2018-04-14,2018-04-16,3
2,-0CMnp02rNY,CANADA::-0CMnp02rNY,2018-06-05,2018-06-05,2018-06-05,1
3,-0DjA_r32uQ,CANADA::-0DjA_r32uQ,2018-02-14,2018-02-14,2018-02-14,1
4,-0F7AFzWXik,CANADA::-0F7AFzWXik,2018-02-19,2018-02-19,2018-02-19,1


row-level shape: (161470, 28)
video-level shape: (63805, 30)
row-level trending date range: 2017-11-14 00:00:00 to 2018-06-14 00:00:00


### Analysis

The cleaned `clean_df` dataframe is now ready for descriptive analysis, and `video_level_df` is ready for per-video summaries. The notebook keeps both views in memory so the raw trending history is still available for date-based analysis.

In [12]:
display(clean_df.describe(include='all').T)
display(clean_df[['views', 'likes', 'dislikes', 'comment_count', 'engagement_rate', 'like_ratio', 'days_to_trend', 'tag_count', 'days_trending']].describe().T)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
index,161470.0,NaN,NaN,NaN,80734.5,0.0,40367.25,80734.5,121101.75,161469.0,46612.51832
video_id,161470,55886,unknown,1799,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,161470,NaN,NaN,NaN,2018-02-26 05:14:01.374868,2017-11-14 00:00:00,2018-01-03 00:00:00,2018-02-25 00:00:00,2018-04-23 00:00:00,2018-06-14 00:00:00,NaN
title,161470,56905,Childish Gambino - This Is America (Official V...,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,161470,12361,The Late Show with Stephen Colbert,653,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,161470.0,<NA>,<NA>,<NA>,19.461151,1.0,15.0,23.0,24.0,44.0,7.432001
publish_date,161470,NaN,NaN,NaN,2018-02-11 12:09:47.522140,2006-07-23 00:00:00,2017-12-29 00:00:00,2018-02-19 00:00:00,2018-04-16 00:00:00,2018-06-14 00:00:00,NaN
time_frame,161470,24,16:00 to 16:59,16477,NaN,NaN,NaN,NaN,NaN,NaN,NaN
published_day_of_week,161470,7,Friday,28622,NaN,NaN,NaN,NaN,NaN,NaN,NaN
publish_country,161470,4,US,40949,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
views,161470.0,2419854.095739,10437486.871643,223.0,101538.25,384739.5,1339528.0,424538912.0
likes,161470.0,65661.944076,226061.669131,0.0,1975.0,9840.0,40062.75,5613827.0
dislikes,161470.0,3490.152883,31147.788966,0.0,85.0,348.0,1350.0,1944971.0
comment_count,161470.0,7035.493702,34041.213581,0.0,279.0,1144.0,4144.75,1626501.0
engagement_rate,161470.0,0.041603,0.03718,0.0,0.014351,0.031276,0.057489,0.528285
like_ratio,160032.0,0.931537,0.099103,0.0,0.923404,0.96585,0.983277,1.0
days_to_trend,161470.0,14.711271,145.192259,0.0,1.0,3.0,7.0,4215.0
tag_count,161470.0,17.850691,12.476725,0.0,8.0,16.0,26.0,124.0
days_trending,161470.0,12.992705,49.331451,1.0,2.0,4.0,12.0,530.0


## Summary

- The raw dataset was loaded, inspected, and cleaned in memory.
- Date, numeric, and boolean fields were normalized to analysis-friendly types.
- Placeholder text values were handled explicitly instead of being left in the dataset.
- A row-level dataframe and a per-video dataframe were created for different analysis needs.
- The requested engineered features are available in `clean_df`.

## Recommendations / Conclusion

- Use `clean_df` for time-based analysis of trending days and publishing patterns.
- Use `video_level_df` when the question is about the latest snapshot per video.
- Treat `engagement_rate` and `like_ratio` as derived metrics with zero-denominator protection.
- Inspect `metric_issue_flag` before drawing conclusions from any row that has inconsistent metrics.
- Keep the notebook as the source of truth for the cleaning logic; do not rely on a hidden export.